# Notebook 8 - Subscriber Health Scoring

**Goal** - classify all 1,082,190 subscribers as Red, Amber, or Green based on their churn probability score.


## Load scores and set thresholds

Red ≥ 0.70 — Amber ≥ 0.30 — Green < 0.30

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED = Path(r"C:\kkbox-retention-analytics\data\processed")

df = pd.read_csv(PROCESSED / "master_scored.csv")

RED_THRESHOLD   = 0.70
AMBER_THRESHOLD = 0.30

print(f"Subscribers loaded: {df.shape[0]:,}")
print(f"\nThresholds:")
print(f"  Red:   churn_probability >= {RED_THRESHOLD}")
print(f"  Amber: churn_probability >= {AMBER_THRESHOLD} and < {RED_THRESHOLD}")
print(f"  Green: churn_probability <  {AMBER_THRESHOLD}")

Subscribers loaded: 1,082,190

Thresholds:
  Red:   churn_probability >= 0.7
  Amber: churn_probability >= 0.3 and < 0.7
  Green: churn_probability <  0.3


## Assign health labels

In [3]:
import numpy as np 
conditions = [
    df['churn_probability'] >= RED_THRESHOLD,
    df['churn_probability'] >= AMBER_THRESHOLD
]
labels = ['Red', 'Amber', 'Green']

df['health_status'] = np.select(conditions, labels[:-1], default='Green')

counts = df['health_status'].value_counts()
shares = df['health_status'].value_counts(normalize=True)

print(f"{'Status':<8} {'Count':>10} {'Share':>8}")
print("-" * 28)
for status in ['Red', 'Amber', 'Green']:
    print(f"{status:<8} {counts[status]:>10,} {shares[status]:>8.1%}")

Status        Count    Share
----------------------------
Red         145,143    13.4%
Amber        79,935     7.4%
Green       857,112    79.2%


## Behavioural profile per segment

Shows what each group actually looks like - not just the churn rate, but the underlying signals that got them there.

In [6]:
master_fe = pd.read_csv(PROCESSED / "master_fe.csv")
df_full = df.merge(master_fe.drop(columns=['is_churn']), on='msno', how='left')

MONTHLY_PRICE = 149

summary = (
    df_full.groupby('health_status')
    .agg(
        subscribers      = ('msno',              'count'),
        avg_churn_prob   = ('churn_probability', 'mean'),
        actual_churn_rate= ('is_churn',          'mean'),
        avg_plan_commit  = ('plan_commitment',   'mean'),
        avg_recency      = ('recency_ratio',     'mean'),
        avg_cancel_rate  = ('cancel_rate',       'mean'),
    )
    .round(3)
)

summary['monthly_revenue_at_risk'] = (
    df_full.groupby('health_status')
    .apply(lambda x: (x['churn_probability'] * MONTHLY_PRICE).sum(), include_groups=False)
    .round(0)
    .astype(int)
)

summary = summary.loc[['Red', 'Amber', 'Green']]
print(summary.to_string())

               subscribers  avg_churn_prob  actual_churn_rate  avg_plan_commit  avg_recency  avg_cancel_rate  monthly_revenue_at_risk
health_status                                                                                                                        
Red                 145143           0.896              0.597            8.560        0.071            0.045                 19384573
Amber                79935           0.578              0.120            9.714        0.014            0.027                  6879527
Green               857112           0.031              0.003           29.752        0.052            0.012                  4010008


## Save

In [7]:
output_cols = ['msno', 'is_churn', 'churn_probability', 'health_status']
df[output_cols].to_csv(PROCESSED / "master_health.csv", index=False)

summary.to_csv(PROCESSED / "health_summary.csv")

verify = pd.read_csv(PROCESSED / "master_health.csv")
print(f"master_health.csv:  {verify.shape[0]:,} rows × {verify.shape[1]} columns")
print(f"health_summary.csv: {summary.shape[0]} rows × {summary.shape[1]} columns")
print(f"\nHealth distribution:")
print(verify['health_status'].value_counts())

master_health.csv:  1,082,190 rows × 4 columns
health_summary.csv: 3 rows × 7 columns

Health distribution:
health_status
Green    857112
Red      145143
Amber     79935
Name: count, dtype: int64


## Conclusion

1,082,190 subscribers classified:
- **Red** - 145,143 (13.4%) — churn probability > 0.70
- **Amber** - 79,935 (7.4%) — churn probability 0.30–0.70
- **Green** - 857,112 (79.2%) — churn probability < 0.30

Plan commitment separates the groups most sharply - Green subscribers show 3.5× higher plan commitment than Red ones.

Actual churn rates validate the model: Red 59.7% - Amber 12.0% - Green 0.3%.